# Session 12 — Change detection and time series
*Séance 12 — Détection de changements et séries temporelles*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noe2019/ML4EO_Bootcamp/blob/main/notebooks/12_changements_series_temporelles.ipynb)

**Course:** Machine Learning for Earth Observation (45 h) · **Instructor:** Noé Carème Fouotsa Manfouo

| Training track · Parcours de formation | This session · Cette séance |
|---|---|
| Geomatics / environment · Géomatique / environnement (UdeS géomatique appliquée; Bishop's Environmental Studies & Geography) | ● full session / séance complète |
| Computer science · Informatique (UdeS informatique; Bishop's Computer Science) | ● full session / séance complète |
| College / continuing education · Cégep / formation continue | — not in track / hors parcours |

**By the end of this session you will be able to:**

1. Decompose an NDVI time series into seasonality and anomalies.
2. Detect drought periods with standardized anomalies.
3. Group pixels with similar temporal behaviour with k-means.
4. Extract a MODIS NDVI series in Earth Engine (optional).

**How to use this notebook:** read the lecture cells, run every code cell (`Shift + Enter`), then complete the cells marked
`# YOUR CODE HERE` in the lab section. Run the ✅ check cells to get instant feedback.

In [ ]:
# --- Setup (run once) -------------------------------------------------------
import sys, pathlib, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not pathlib.Path("ML4EO_Bootcamp").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/noe2019/ML4EO_Bootcamp.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *"geopandas rasterio rioxarray".split()], check=True)
    ROOT = pathlib.Path("ML4EO_Bootcamp")
else:
    ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA = ROOT / "data"
sys.path.append(str(ROOT / "notebooks"))   # course helper module (ml4eo_data.py)


def check(condition, message="Check failed"):
    """Instant feedback: raises an error with a hint if the condition is False."""
    assert condition, message
    print("✅ Passed")


def check_equal(value, expected, message=None):
    assert value == expected, message or f"Expected {expected!r}, got {value!r}"
    print("✅ Passed")

## Part A — Lecture

Many questions are about **change**: did crops suffer from the 2015–2016 El Niño drought? Did lockdowns affect farming?
A pixel's NDVI follows a seasonal cycle; we compare each month with its **climatology** (mean of the same month over
all years) to obtain **anomalies**. This session reproduces, with simulated data, the approach of Fouotsa Manfouo & Von Fintel
(2022) on KwaZulu-Natal smallholder and commercial farms.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from ml4eo_data import ndvi_time_series

wide, info = ndvi_time_series()
by_system = wide.T.groupby(info.set_index("pixel")["system"]).mean().T
by_system.plot(figsize=(10, 3), title="Mean monthly NDVI")
plt.axvspan("2015-06-01", "2016-06-01", color="orange", alpha=.2, label="drought"); plt.legend(); plt.show()

In [ ]:
month = wide.index.month
climatology = wide.groupby(month).transform("mean")
std = wide.groupby(month).transform("std")
z = (wide - climatology) / std            # standardized anomaly, per pixel and month
z.T.groupby(info.set_index("pixel")["system"]).mean().T.plot(figsize=(10, 3), title="Mean standardized NDVI anomaly")
plt.axhline(-1, ls="--", c="k"); plt.show()

## Part B — Lab

**Task 1 — Drought months.** A month is in **drought** for a farming system when its mean NDVI is more than **0.1 below
its climatology** (anomaly = NDVI − climatology < −0.1). Build a boolean
DataFrame `drought` (rows = months, columns = systems) and the number of drought months per system `n_drought`.

In [ ]:
# YOUR CODE HERE
drought = None
n_drought = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(n_drought["Smallholder"] > n_drought["Commercial"], "Smallholders should be hit harder")

**Task 2 — Impact.** Compute the mean NDVI loss during the drought (June 2015–June 2016) relative to climatology, per system,
as a percentage (`loss_pct`, negative = loss).

In [ ]:
# YOUR CODE HERE
loss_pct = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(loss_pct["Smallholder"] < loss_pct["Commercial"] < 0)

**Task 3 — Clustering profiles.** Cluster the pixels by their NDVI **anomaly** profiles (`wide - climatology`, rows = pixels)
with `KMeans(2)`, store
labels in `clusters`, and compute `purity` = share of pixels whose cluster matches the majority system of that cluster.

In [ ]:
# YOUR CODE HERE
clusters = None
purity = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(purity > 0.9, "Anomaly profiles should separate the two farming systems")

### Optional — real MODIS NDVI in Earth Engine
Set `USE_GEE = True` (and your project ID) to repeat the analysis with **MOD13Q1** (250 m, 16-day NDVI) over our Western Cape ward.

In [ ]:
USE_GEE = False
if USE_GEE:
    import ee, geemap, geopandas as gpd
    ee.Initialize(project="ee-yourname")
    ward = geemap.geopandas_to_ee(gpd.read_file(DATA / "SA_LEVEL4" / "ADM4_PCODE_ZA1011001.shp")[["geometry"]])
    modis = ee.ImageCollection("MODIS/061/MOD13Q1").filterDate("2010-01-01", "2022-01-01").select("NDVI")
    series = modis.map(lambda img: ee.Feature(None, {
        "date": img.date().format("YYYY-MM-dd"),
        "ndvi": img.reduceRegion(ee.Reducer.mean(), ward.geometry(), 250).get("NDVI")}))
    real = geemap.ee_to_df(ee.FeatureCollection(series)).dropna()
    real["ndvi"] = real["ndvi"] * 0.0001
    real.set_index(pd.to_datetime(real["date"]))["ndvi"].plot(figsize=(10, 3), title="MODIS NDVI, ward ZA1011001")

**❓ Question —** Anomalies were negative for both systems, but much less for commercial farms. Which management factors could explain it, and how would you test them with additional data?

*Your answer here.*

## Summary

- Anomalies against a monthly climatology remove the seasonal cycle.
- Standardized anomalies (z-scores) make pixels and regions comparable.
- Unsupervised clustering of temporal profiles reveals contrasting behaviours without labels.

**To go further:**

- Fouotsa Manfouo, N. C. & Von Fintel, D. (2022). Investigating the effects of drought and lockdowns on smallholder and commercial agricultural production in KwaZulu-Natal using remotely sensed data. *Heliyon*, 8(11).
- Fouotsa Manfouo, N. C. (2024). Monitoring commercial and smallholder production inequalities during South Africa's lockdown using remotely sensed data. *Agrosystems, Geosciences & Environment*, 7(1). doi:10.1002/agg2.20460
- Didan, K. (2021). MODIS/Terra Vegetation Indices 16-Day L3 Global 250 m, v061. NASA LP DAAC.